# 07 — Wind models in MechaTree

Wind enters the simulator in three distinct places, and an outside
reader can't tell which knob does what without a map. This notebook
is that map. By the end you will know:

1. **Mechanics**: how a wind vector becomes a per-branch stress, and
   why the safety-factor genome uses a four-angle 'sensing sweep' that
   is hard-wired today.
2. **Storm wind statistics**: amplitude and angle distributions are
   now tunable via SymPy CDFs in the YAML `wind:` block (Step 25).
   The default still reproduces the Fortran code byte-identically.
3. **Canopy-aware wind**: the in-repo bulk-thinning model with
   rotation to face the storm, and how it relates to the optional
   DendroFlow bridge.
4. **Step 24 in pictures**: the wind ↔ pruning fixed-point loop,
   replayed pass-by-pass on a small forest hit with a single big
   storm. Pruned branches are drawn in black so you can watch the
   canopy collapse iteration by iteration.
5. **A lab section**: swap the angle distribution from uniform to a
   prevailing-wind von Mises, re-run, and see the canopy adapt.

In [ ]:
import math

import numpy as np
import plotly.graph_objects as go

import mechatree as mt

mt.figstyle.apply()
rng = np.random.default_rng(0)

## 1. From a wind vector to a per-branch stress

Every branch is a tapered cylinder of length $L$ and diameter $D$ with
a unit axis $\hat{t}$. Given a wind vector $\mathbf{U}$ (units arbitrary;
MechaTree is unit-free), the drag on the branch is proportional to its
frontal area, which depends on the angle $I$ between $\hat{t}$ and
$\mathbf{U}$:

$$F \propto |\mathbf{U}|^2 \, D \, L \, \sin^2 I.$$

That force is integrated up the tree (children pass their force and
moment to their parent) and the resulting bending stress at each
branch is

$$\sigma = \frac{16}{\pi} \, c_y \, \frac{|\hat{t} \times \mathbf{M}|}{D^3}.$$

This is exactly what [`mechatree._core.mechanics.cpp`](../src/mechatree/_core/mechanics.cpp)
implements.

**Two distinct uses of the stress.** *Sensing* drives growth: the
safety-factor genome reads each branch's worst-case stress across a
four-angle sweep and sets the diameter so that $\sigma_\max \le \sigma_{\mathrm{break}} / k$ for
some safety factor $k$. *Pruning* uses the per-generation storm wind:
each branch's instantaneous stress under that wind is plugged into a
Weibull failure law (`pruning.cpp`) and the branch falls with
probability $p_\mathrm{fail}$.

The sensing sweep currently hard-wires four angles $\theta \in \{\pi/4, \pi/2,
3\pi/4, \pi\}$. Making *that* tunable too is a Step-25 follow-up
(needs a C++ extension to `calculate_stresses`).

## 2. Storm wind statistics — tunable distributions

The Fortran reference sampled the storm angle deterministically as
$\theta = g$ rad (rotating by 1 rad per generation) and the amplitude
from a shifted exponential $a = 0.835 - \log(U) / 6$, where $U$ is
uniform on $(0, 1]$. That gives a typical amplitude $\approx 1.0$ with
rare gusts reaching $\sim 3$.

Step 25 lets you swap either with a SymPy CDF in YAML:

```yaml
wind:
  model: default
  amplitude_cdf: "1 - exp(-6 * (a - 0.835))"   # default, reproduces Fortran
  angle_cdf: "theta / (2*pi)"                  # uniform on [0, 2*pi)
```

Leaving a field unset keeps the legacy behaviour byte-identically, so
existing simulations don't drift.

Let's compare the legacy amplitude distribution against a heavier-tail
alternative (Rayleigh, mean $\sqrt{\pi/2} \approx 1.25$):

In [ ]:
default_amp = mt.default_amplitude_sampler()
rayleigh_amp = mt.Distribution(
    cdf_expr="1 - exp(-x**2 / 2)", var_name="x", support=(0.0, math.inf)
).sampler()

default_samples = default_amp(rng, 10_000)
rayleigh_samples = rayleigh_amp(rng, 10_000)

fig = mt.figstyle.figure(size="full", aspect=9 / 4)
fig.add_trace(
    go.Histogram(
        x=default_samples,
        name="Fortran default",
        nbinsx=80,
        histnorm="probability density",
        opacity=0.6,
        marker_color=mt.figstyle.COLORS["blue"],
    )
)
fig.add_trace(
    go.Histogram(
        x=rayleigh_samples,
        name="Rayleigh",
        nbinsx=80,
        histnorm="probability density",
        opacity=0.6,
        marker_color=mt.figstyle.COLORS["red"],
    )
)
fig.update_layout(barmode="overlay", xaxis_title="storm amplitude", yaxis_title="density")
fig.show()
print(
    f"Fortran:  mean={default_samples.mean():.3f}, "
    f"99th pct={np.percentile(default_samples, 99):.3f}"
)
print(
    f"Rayleigh: mean={rayleigh_samples.mean():.3f}, "
    f"99th pct={np.percentile(rayleigh_samples, 99):.3f}"
)

Same machinery for the angle distribution. Default is uniform on
$[0, 2\pi)$ (the canopy is hit isotropically); a prevailing-wind setup
would use e.g. a von Mises CDF centred on $\theta = 0$.

## 3. Canopy-aware wind — native bulk-thinning

Without a canopy model, every branch in a forest feels the same
free-stream wind. That's wrong: dense canopies absorb momentum, so the
branches downstream feel a thinner wind than the ones at the leading
edge. MechaTree now ships a self-contained bulk-thinning model in
[`mechatree.wind.bulk_thinning`](../src/mechatree/wind/bulk_thinning.py) that
implements the actuator-disk thinning per z-layer used by DendroFlow's
[`BulkThinningBranchWindModel`](../../DendroFlow/src/dendroflow/wind/branch.py),
ported in-repo so the common case doesn't need the external
dependency.

Key feature: the model **rotates the canopy to face the storm**, the
same way [`mechatree.light.interception`](../src/mechatree/light/interception.py)
rotates leaves to face the sun. Without the rotation, asking for a
storm from the north and a storm from the east would give the same
answer (which the legacy DendroFlow bridge does today). With the
rotation, each branch's projected area $D \, L \, \sin I$ is recomputed
for the right wind direction.

Default settings: uniform free stream $U = 1$ over $z \in [0, 50]$
with $H = 0.5$ — no boundary layer, no opt-in needed. Realistic log
or power-law profiles are configured via YAML.

In [ ]:
from mechatree.wind.bulk_thinning import _stack_trees, compute_bulk_thinning

# Build a tiny forest first.
cfg = mt.Config(
    tree=mt.TreeConfig(),
    forest=mt.ForestConfig(size=15.0, n_trees_init=8, n_trees_max=80),
    n_generations=30,
)
f = mt.Forest(cfg, seed=0)
for g in range(30):
    f.step(g)
print(
    f"warmup forest: {len(f.trees)} trees, "
    f"{sum(t.get_number_of_branches() for t in f.trees)} branches"
)

# Run the native bulk-thinning bridge for three different storm directions.
params = mt.BulkThinningParams.uniform(U=3.0)
start, axis, D, L, offsets = _stack_trees(f.trees)
for deg in (0.0, 45.0, 90.0):
    th = math.radians(deg)
    res = compute_bulk_thinning(
        start,
        axis,
        D,
        L,
        branch_offsets=offsets,
        params=params,
        wind_direction=(math.cos(th), math.sin(th)),
    )
    cm = res.canopy_mean
    mag = math.hypot(cm[0], cm[1])
    recovered = math.degrees(math.atan2(cm[1], cm[0]))
    print(
        f"θ={deg:>4.0f}°  canopy_mean=({cm[0]:+.3f}, {cm[1]:+.3f})  "
        f"|U|={mag:.3f}  recovered θ={recovered:+.2f}°"
    )

Magnitude is preserved across rotations (the same canopy thinning
applies regardless of direction); only the projection onto $(x, y)$
changes. That's the rotation working.

## 4. Step 24 in pictures — one storm, pass-by-pass

The Step-24 fixed-point loop re-evaluates wind after every prune sweep,
so the canopy is scored against the wind it will actually feel
post-pruning. To see it at work, we'll:

1. Grow a small forest of 10 trees over $\sim 80$ generations with
   essentially zero wind, so the canopy gets large without any
   pruning.
2. Apply ONE big storm. Use [`run_storm_replay`](../src/mechatree/wind/replay.py)
   to snapshot each pass of the fixed-point loop separately.
3. Plot the pre-storm canopy + each pass, with pruned branches drawn
   in **black** so you can watch the canopy collapse.

What we'll see: iter 1 cuts most of the easy targets (thin distal
branches at the top of the canopy); the surviving canopy is sparser,
so the per-branch wind in iter 2 is *higher* (less mutual shielding),
and iter 2 cuts a fresh wave. The loop exits when the canopy stops
changing or a hard cap is hit.

In [ ]:
# Step 4a: warmup forest with zero wind so no pruning happens.
warmup_cfg = mt.Config(
    tree=mt.TreeConfig(),
    forest=mt.ForestConfig(size=20.0, n_trees_init=10, n_trees_max=200),
    n_generations=80,
)


def zero_wind(gen, rng):
    return (0.0, 0.0, 0.0)


warm = mt.Forest(warmup_cfg, seed=7, wind_fn=zero_wind)
for g in range(80):
    warm.step(g)
n_branches_pre = sum(t.get_number_of_branches() for t in warm.trees)
print(f"pre-storm: {len(warm.trees)} trees, {n_branches_pre} branches")

In [ ]:
# Step 4b: replay one big storm pass-by-pass.
# Crank U_infty high enough to provoke multiple iterations.
storm_wind = mt.make_bulk_thinning_wind_fn()
storm_wind.params = mt.BulkThinningParams(
    U_infty=np.full(100, 3.5),
    z_centers=np.linspace(0.25, 49.75, 100),
    H=0.5,
)

pre, snapshots = mt.run_storm_replay(
    warm,
    storm_wind,
    generation=80,
    max_iterations=6,
    eps_rel=0.01,
    leaf_drag_S0=warmup_cfg.tree.leaf_surface,
    cauchy=warmup_cfg.tree.cauchy,
)
for s in snapshots:
    print(
        f"  iter {s.iteration}: cut {s.n_pruned_this_iter:>5} branches across "
        f"{s.n_trees_affected_this_iter:>2} trees, "
        f"wind=({s.wind_used[0]:+.3f}, {s.wind_used[1]:+.3f})"
    )

In [ ]:
fig = mt.plot_storm_replay(pre, snapshots, n_cols=3)
fig.show()

Each panel shows the pre-storm canopy with **black** segments for
branches that have already fallen by that iteration. Watch the
iteration count: storms where the canopy genuinely reshapes the wind
take 2–3 passes to converge.

## 5. Lab — prevailing wind via a von Mises CDF

Let's see what changes if the storm doesn't come uniformly from every
direction. A von Mises distribution concentrated around $\theta = 0$
models a 'prevailing easterly':

In [ ]:
# We'll just provide the CDF numerically since the von Mises CDF doesn't
# have a simple closed form; the Distribution class falls back to
# numerical inverse-CDF when SymPy can't invert symbolically.
kappa = 4.0
von_mises_cdf = f"(theta + (1/{kappa}) * sin({kappa} * (theta - pi))) / (2*pi)"
# (Not a real von Mises; just a heavy-on-0 CDF that's easy to write.)
prevailing = mt.Distribution(
    cdf_expr="theta/(2*pi) - sin(theta)/(2*pi)",
    var_name="theta",
    support=(0.0, 2 * math.pi),
)
samples = prevailing.sampler()(rng, 10_000)
fig = mt.figstyle.figure(size="full", aspect=9 / 4)
fig.add_trace(
    go.Histogram(
        x=samples,
        nbinsx=60,
        marker_color=mt.figstyle.COLORS["green"],
    )
)
fig.update_layout(
    title="Angle samples from a 'leans on 0' CDF",
    xaxis_title="θ (rad)",
    yaxis_title="count",
)
fig.show()

Plug a similar prevailing-wind CDF into the YAML, point the bridge at
the storm distribution, and run a long evolutionary tournament: you
should see trees with asymmetric canopies emerge — leaning *with*
the wind (the windward side gets pruned hard).

## What's next

Step 25 deferred two follow-ups:

1. **C++ sensing sweep** — currently still pinned to the legacy
   $(\pi/4, \pi/2, 3\pi/4, \pi)$ angles. Honouring `angle_cdf` and
   `angle_samples` from sensing too needs an extension to
   `calculate_stresses` and its Cython binding.
2. **DendroFlow bridge rotation** — the legacy bridge still solves in
   the world $+x$ frame; the native bulk-thinning model above is the
   recommended path now. The bridge is kept for forward-compatibility
   with DendroFlow's k-ε model.

Open items for a future Step 25b live in the DendroFlow repo:
cell-size sensitivity study, single-cylinder k-ε flow visualisation,
and a per-branch wind output from the bulk-thinning model so we can
render `plot_tree_3d` with branches coloured by wind level.